<a href="https://colab.research.google.com/github/johannm00-ux/Deep-Learning-Project-1/blob/main/First_Project_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDU Deep Learning (DM873 / DS809) — Project 1
## Cat & Dog Classification with Convolutional Neural Networks

**Due Date:** Friday 06-11-2026 at 23:59  
**Submission format:** Single PDF uploaded to Itslearning (Report + Code + Visualizations)

---
### Group Work & Contribution Statement
*As required by the assignment brief, individual contributions must be explicitly stated:*
* **Member 1 (Johann Birk Skaaning Mogensen):** Data pipeline, augmentation design, and model architecture.
* **Member 2:** Hyperparameter exploration, regularization experiments, and training loops.
* **Member 3:** Evaluation, confusion matrix, error analysis, and feature map visualizations.
* **Member 4:** Report synthesis, literature justification, and competition compliance.
*(Update names and assigned roles as agreed with your team)*


## 0. Setup & High-Speed Dataset Extraction
This cell automatically detects whether you are running in **Google Colab** or **locally**.
* In Colab, it downloads and extracts the dataset directly to the local fast SSD (`/content/catdog_data`), preventing network bottlenecks from Google Drive.
* Automatically selects GPU (`cuda`), Apple Silicon (`mps`), or CPU.


In [1]:
import os
import sys
import shutil
import zipfile
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Set reproducible seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Detect compute device: CUDA GPU > Apple Silicon MPS > CPU
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Using compute device: {device}")

# Dataset setup
DATA_DIR = './catdog_data'
if 'google.colab' in sys.modules:
    DATA_DIR = '/content/catdog_data'
    if not os.path.exists(DATA_DIR):
        print("Running on Google Colab. Downloading catdog_data.zip from GitHub Release...")
        !wget -q https://github.com/johannm00-ux/Deep-Learning-Project-1/releases/download/v1.0/catdog_data.zip -O /content/catdog_data.zip
        with zipfile.ZipFile('/content/catdog_data.zip', 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print("Extraction complete!")

print(f"Dataset root: {DATA_DIR}")
cats_dir = os.path.join(DATA_DIR, 'cats')
dogs_dir = os.path.join(DATA_DIR, 'dogs')
cat_files = [os.path.join(cats_dir, f) for f in os.listdir(cats_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
dog_files = [os.path.join(dogs_dir, f) for f in os.listdir(dogs_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Found {len(cat_files)} cat images and {len(dog_files)} dog images (Total: {len(cat_files) + len(dog_files)})")


Using compute device: cpu
Running on Google Colab. Downloading catdog_data.zip from GitHub Release...
Extraction complete!
Dataset root: /content/catdog_data
Found 1200 cat images and 1200 dog images (Total: 2400)
